# Calculating regression coefficients for individual conversations in a given dataset

In [1]:
import os
import numpy as np
import pandas as pd
from scipy.linalg import lstsq
from scipy.stats import t as tdist

import pyarrow.parquet as pq
from tqdm.notebook import tqdm
from datetime import datetime as dt
# import statsmodels.formula.api as smf
import warnings;warnings.filterwarnings('ignore')

In [2]:
OUTPUTS_NAME = 'results-{}-updated.csv'

In [3]:
# pre recent run
# DATA_PATH = '../data/ckpts/rosen'

DATA_PATH = '../../updated-data/obs/lme-ready'
NULL_DATA_PATH = '../../updated-data/null/null-lme-ready'

OUTPUTS_PATH = 'reports'
if not os.path.exists(OUTPUTS_PATH):
    os.mkdir(OUTPUTS_PATH)

In [4]:
def grab_all_files(PATH, file_type='.parquet'):
    files = [
        [
            os.path.join(root, f) for f in files
            if f.endswith(file_type) and (not f.startswith('.'))
        ]
        for root, _, files in os.walk(PATH)
    ]
    return sum(files, [])

In [6]:
fs = [
    f for f in os.listdir(DATA_PATH)
    if (('xling-' in f) or ('callfriend' in f) or ('callhome' in f))
    and (not f.startswith('.'))
    and ('yid' not in f)
    and f.endswith('.parquet')
]
len(fs)

1262

## Processing individual datasets

In [7]:
Y_VAR = 'I'

In [8]:
param_list = [
    'nx',
    'ny'
]
processed_params = [
    'Intercept',
    'self',
    'other',
    # 'null',
    'turn_delta_self',
    'turn_delta_other'
]

In [9]:
df_scale, df_regression = [], []

#### Crunching numbers one individual speaker at a time

In [10]:
for f in tqdm(fs):
    # print(f.split('/')[-1])

    corpus = f.split('/')[-1].split('-')[0]
    cid = '-'.join(f.split('/')[-1].split('-')[1:])

    dfo = pq.ParquetFile(os.path.join(DATA_PATH, f))
    if f.startswith('grace-'):
        dfn = pq.ParquetFile(os.path.join(NULL_DATA_PATH, f.replace('grace', 'Miao-fNIRS')))
    else:
        dfn = pq.ParquetFile(os.path.join(NULL_DATA_PATH, f))

    df = [dfo.read(columns=[Y_VAR]+param_list+['x_speaker', 'y_speaker', 'x_turn_id', 'y_turn_id']).to_pandas()]
    df[-1]['null'] = False

    df += [dfn.read(columns=[Y_VAR]+param_list+['x_speaker', 'y_speaker', 'x_turn_id', 'y_turn_id']).to_pandas()]
    df[-1]['null'] = True
    df = pd.concat(df, ignore_index=True)

    # table = pq.ParquetFile(f)
    #
    # df = table.read(columns=[Y_VAR]+param_list+['x_speaker', 'y_speaker', 'x_turn_id', 'y_turn_id']).to_pandas()

    # dealing with temporal distance between turns
    df['turn_delta'] = df['y_turn_id'] - df['x_turn_id']
    df = df.loc[
        (df['turn_delta'] > 0)
        & (df['turn_delta'] <= 200)
    ]
    df['turn_delta'].loc[df['null']] = 0.
    df['turn_delta'] += 1
    df['turn_delta'] = np.log(df['turn_delta'])


    # setting up self and other cols
    df['self'] = (df['x_speaker'] == df['y_speaker'])
    df['other'] = ~df['self']

    # how we treat self and oter in null condition
    df['self'].loc[df['null']] = False
    df['other'].loc[df['null']] = False

    df['turn_delta_other'] = df['turn_delta']
    df['turn_delta_self'] = df['turn_delta']
    df['turn_delta_other'].loc[df['self']] = 0.
    df['turn_delta_self'].loc[df['other']] = 0.

    # convert T/F values to integers
    df['self'] = df['self'].astype(int)
    df['other'] = df['other'].astype(int)
    df['null'] = df['null'].astype(int)

    df['Intercept'] = 1.

    params, _, _, _ = lstsq(
        df[param_list + processed_params].values,
        df[Y_VAR].values
    )

    df_regression += [
        {
            'corpus': corpus,
            'cid': cid,
            'param': name,
            'beta': param,
            'k': len(df),
            'nspeakers': df['x_speaker'].nunique()
        }

    for name, param in list(zip(param_list + processed_params, params))]

df_regression = pd.DataFrame(df_regression)

  0%|          | 0/1262 [00:00<?, ?it/s]

In [11]:
df_regression['lang'] = [cid.split('-')[1] for cid in df_regression['cid']]
df_regression['lang'] = ['eng' if (lang.endswith('.parquet') or (lang == 'callfriend')) else lang for lang in df_regression['lang']]

#### Last checks and saving outputs

In [ ]:
df_regression.head()

In [12]:
df_regression['lang'].unique()

array(['deu', 'fin', 'cro', 'ko', 'eng', 'spa', 'fra', 'zho'],
      dtype=object)

In [ ]:
# df_regression.isna().sum()

In [ ]:
# df_regression.to_csv(
#     os.path.join(OUTPUTS_PATH, str(OUTPUTS_NAME).format(dt.now().strftime("%Y%m%d%H%M"))),
#     index=False,
#     encoding='utf-8'
# )

## Aggregate Results

In [13]:
res = [
    {
        'param': param,
        'beta': df_regression['beta'].loc[df_regression['param'].isin([param])].mean(),
        'sd': df_regression['beta'].loc[df_regression['param'].isin([param])].std(),
        'k': df_regression['param'].isin([param]).sum()
    } for param in df_regression['param'].unique()
]

res = pd.DataFrame(res)

In [14]:
res['se'] = res['sd'] / np.sqrt(res['k'])
res['t'] = res['beta'] / res['se']
res['p'] = [1-tdist.cdf(t.__abs__(), dof-1).item() for t,dof in res[['t', 'k']].values]

In [15]:
res.head(n=10)

,param,beta,sd,k,se,t,p
0,nx,0.151192,0.017783,1262,0.000501,302.040363,0.000000
1,ny,-0.022630,0.007345,1262,0.000207,-109.449646,0.000000
2,Intercept,-0.167786,0.173708,1262,0.004890,-34.313529,0.000000
3,self,-0.147235,0.145429,1262,0.004094,-35.965564,0.000000
4,other,0.015345,0.159116,1262,0.004479,3.426082,0.000316
5,turn_delta_self,0.024752,0.048605,1262,0.001368,18.091019,0.000000
6,turn_delta_other,-0.002340,0.043305,1262,0.001219,-1.919652,0.027564


#### Save results

In [16]:
res.to_csv(
    os.path.join(OUTPUTS_PATH, 'agg-params-no-yiddish.csv'),
    index=False,
    encoding='utf-8'
)

## Language level analysis of results

In [17]:
split_by = ['lang','param']

In [16]:
df0 = df_regression[split_by+['beta']].groupby(by=split_by).agg('mean').reset_index()
df0['std'] = df_regression[split_by+['beta']].groupby(by=split_by).agg('std').reset_index()['beta'].values

df0['k'] = df_regression[split_by+['cid']].groupby(by=split_by).agg('count').reset_index()['cid'].values
# df0['k'] = df_regression[split_by+['k']].groupby(by=split_by).agg('sum').reset_index()['k'].values

df0['se'] = df0['std'] / np.sqrt(df0['k'])

In [17]:
df0['t'] = df0['beta'] / df0['se']

In [18]:
df0['p'] = [1-tdist.cdf(t.__abs__(), dof-1).item() for t,dof in df0[['t', 'k']].values]

In [19]:
df0.head(n=75)

,lang,param,beta,std,k,se,t,p
0,cro,Intercept,-0.284309,0.060883,164,0.004754,-59.802075,0.000000
1,cro,nx,0.155759,0.006656,164,0.000520,299.668186,0.000000
2,cro,ny,-0.017729,0.003911,164,0.000305,-58.048706,0.000000
3,cro,other,-0.014714,0.042856,164,0.003346,-4.396834,0.000010
4,cro,self,-0.074529,0.045930,164,0.003587,-20.780141,0.000000
...,...,...,...,...,...,...,...,...
58,zho,ny,-0.027444,0.005347,200,0.000378,-72.580227,0.000000
59,zho,other,0.022172,0.095648,200,0.006763,3.278186,0.000617
60,zho,self,-0.142703,0.096682,200,0.006836,-20.873764,0.000000
61,zho,turn_delta_other,-0.004302,0.016470,200,0.001165,-3.693888,0.000143


In [20]:
df0.to_csv(
    os.path.join(OUTPUTS_PATH, 'by-language-results.csv'),
    index=False, encoding='utf-8'
)

## Wald procedure

In [18]:
from shared.wald_test import wald

In [19]:
X = np.concat([df_regression['beta'].loc[df_regression['param'].isin([param])].values.reshape(-1,1) for param in param_list+processed_params], axis=-1)
X.shape

(1262, 7)

In [20]:
W = wald(X)
W.beta.shape

(7,)

In [21]:
print(param_list+processed_params)

['nx', 'ny', 'Intercept', 'self', 'other', 'turn_delta_self', 'turn_delta_other']


### Self versus other model

In [22]:
contrast = np.array([[0,0,0,1,-1,1,-1]])

contrast @ W.beta, W.test(contrast)

(array([-0.13548769]), (np.float64(1090.4976127206564), 1, np.float64(0.0)))

### Self model

In [26]:
contrast = np.array([[0,0,0,1,0,1,0]])

contrast @ W.beta, W.test(contrast)

(array([-0.26190263]),
 (np.float64(40.12314367897187), 1, np.float64(2.3844715091314583e-10)))

### Other model

In [27]:
contrast = np.array([[0,0,0,0,1,0,1]])

contrast @ W.beta, W.test(contrast)

(array([-0.00280218]),
 (np.float64(0.046984126824487694), 1, np.float64(0.8283967718388133)))

## Other statistics

In [30]:
dfstats = df_regression[['lang', 'cid']].drop_duplicates(subset=['lang', 'cid']).groupby(by=['lang']).agg('count').reset_index()

In [31]:
# kspeakers
s = df_regression[['lang', 'cid', 'nspeakers']].drop_duplicates(subset=['lang', 'cid'])
dfstats['speakers'] = s[['lang', 'nspeakers']].groupby(by=['lang']).agg('sum').reset_index()['nspeakers']
dfstats.head()

,lang,cid,speakers
0,cro,164,595
1,deu,120,260
2,eng,216,465
3,fin,85,170
4,fra,60,136


In [32]:
# comparisons
s = df_regression[['lang', 'cid', 'k']].drop_duplicates(subset=['lang', 'cid'])
dfstats['comparisons'] = s[['lang', 'k']].groupby(by=['lang']).agg('sum').reset_index()['k']
dfstats.head()

,lang,cid,speakers,comparisons
0,cro,164,595,9404696
1,deu,120,260,4879694
2,eng,216,465,12405864
3,fin,85,170,106930
4,fra,60,136,7911017


In [33]:
dfstats.to_csv(
    os.path.join(OUTPUTS_PATH, 'corpus-descriptive-dfstats.csv'),
    index=False,
    encoding='utf-8'
)

In [34]:
dfstats['comparisons'].sum()

np.int64(78287660)

In [35]:
dfstats['cid'].sum()

np.int64(1289)